In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Subset, DataLoader
from sklearn.model_selection import StratifiedKFold
import numpy as np


from dataset import create_dataloaders
from tcn_model import MEGTCN
from train import train_one_epoch
from evaluate import evaluate
from grid_search import run_grid_search

In [2]:
DATA_DIR = "preprocessed_data/Intra"
BATCH_SIZE = 8
EPOCHS = 30
LEARNING_RATE = 1e-3
NUM_CLASSES = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


In [3]:
train_loader, test_loader = create_dataloaders(DATA_DIR, BATCH_SIZE)

Loading test data...
Loading train data...
Loaded 32 training samples
Loaded 8 test samples
Class distribution in training: [8 8 8 8]
Class distribution in test: [2 2 2 2]
Train batches: 4
Test batches: 1


In [ ]:

# K-fold cross-validation with stratification
dataset = train_loader.dataset
k_folds = 4

# Get labels from dataset
labels = np.array([dataset[i][1] for i in range(len(dataset))])

# Stratified k-fold split
skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(dataset)), labels)):
    fold_train_loader = DataLoader(Subset(dataset, train_idx), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(Subset(dataset, val_idx), batch_size=BATCH_SIZE, shuffle=False)

    model = MEGTCN(num_classes=NUM_CLASSES).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    print(f"Starting fold {fold+1}/{k_folds}")
    for epoch in range(EPOCHS):
        train_loss, train_acc = train_one_epoch(model, fold_train_loader, criterion, optimizer, DEVICE)
        val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)

        print(
            f"Fold {fold+1} | Epoch {epoch+1}/{EPOCHS} | "
            f"Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%"
        )


Starting fold 1/4
Number of testing samples in fold 1: 8
Fold 1 | Epoch 1/30 | Train Acc: 50.00% | Val Acc: 87.50%
Number of testing samples in fold 1: 8
Fold 1 | Epoch 2/30 | Train Acc: 87.50% | Val Acc: 100.00%
Number of testing samples in fold 1: 8
Fold 1 | Epoch 3/30 | Train Acc: 100.00% | Val Acc: 100.00%
Number of testing samples in fold 1: 8
Fold 1 | Epoch 4/30 | Train Acc: 87.50% | Val Acc: 100.00%
Number of testing samples in fold 1: 8
Fold 1 | Epoch 5/30 | Train Acc: 91.67% | Val Acc: 100.00%
Number of testing samples in fold 1: 8
Fold 1 | Epoch 6/30 | Train Acc: 100.00% | Val Acc: 100.00%
Number of testing samples in fold 1: 8
Fold 1 | Epoch 7/30 | Train Acc: 100.00% | Val Acc: 100.00%
Number of testing samples in fold 1: 8
Fold 1 | Epoch 8/30 | Train Acc: 100.00% | Val Acc: 100.00%
Number of testing samples in fold 1: 8
Fold 1 | Epoch 9/30 | Train Acc: 95.83% | Val Acc: 100.00%
Number of testing samples in fold 1: 8
Fold 1 | Epoch 10/30 | Train Acc: 95.83% | Val Acc: 100.00

KeyboardInterrupt: 

In [ ]:
run_grid_search(train_loader, test_loader, NUM_CLASSES, epochs=15)


Testing parameters:
{'learning_rate': 0.001, 'kernel_size': 3, 'dropout': 0.2, 'hidden_channels': 32}
Epoch 1/15 | Train Acc: 50.00% | Val Acc: 50.00%
Epoch 2/15 | Train Acc: 90.62% | Val Acc: 50.00%
Epoch 3/15 | Train Acc: 100.00% | Val Acc: 100.00%
Epoch 4/15 | Train Acc: 93.75% | Val Acc: 100.00%
Epoch 5/15 | Train Acc: 100.00% | Val Acc: 100.00%
Epoch 6/15 | Train Acc: 100.00% | Val Acc: 100.00%
Epoch 7/15 | Train Acc: 100.00% | Val Acc: 100.00%
Epoch 8/15 | Train Acc: 100.00% | Val Acc: 100.00%
Epoch 9/15 | Train Acc: 100.00% | Val Acc: 100.00%
Epoch 10/15 | Train Acc: 90.62% | Val Acc: 100.00%
Epoch 11/15 | Train Acc: 100.00% | Val Acc: 100.00%
Epoch 12/15 | Train Acc: 93.75% | Val Acc: 100.00%
Epoch 13/15 | Train Acc: 100.00% | Val Acc: 100.00%
Epoch 14/15 | Train Acc: 100.00% | Val Acc: 100.00%
Epoch 15/15 | Train Acc: 90.62% | Val Acc: 100.00%

Testing parameters:
{'learning_rate': 0.001, 'kernel_size': 3, 'dropout': 0.2, 'hidden_channels': 64}
Epoch 1/15 | Train Acc: 78.12% 